In [ ]:
# CONFIGURATION - edit these values before running the notebook
from pathlib import Path
import datetime, os
import random
import numpy as np
import torch

# Reproducibility
GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
torch.cuda.manual_seed_all(GLOBAL_SEED)

# timestamp for output folders
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# Data paths (placeholders — edit to point at your local copy of the dataset/splits)
base_dir = Path(r'/path/to/dataset_root')
train_csv = Path(r'train_merged.csv')
val_csv   = Path(r'val_merged.csv')
test_csv  = Path(r'test_merged.csv')

# Output paths (placeholder — edit to your desired output location)
output_root = Path(r'/path/to/output/saved_aes')
results_dir = output_root / f'results_{timestamp}'
examples_dir = results_dir / 'examples_test'

# Caching
use_mem_cache = True
use_disk_cache = False
disk_cache_dir = output_root / 'image_cache'

# Training hyperparameters
img_size = 224
batch_size = 32
num_workers = 0
tiny_epochs = 25
vae_epochs = 25
lr_tiny = 1e-3
lr_vae = 1e-3
recon_tol = 0.1

# Early stopping (config)
early_stopping = False        # set True to enable early stopping
es_patience = 3               # epochs with no improvement before stop
es_min_delta = 1e-4           # minimum absolute improvement to reset patience
es_restore_best = True  

# quick-sample option (0.0 < sample_fraction < 1.0 to sample a small subset of each split)
sample_fraction = 0.75
subset_random_seed = GLOBAL_SEED

# Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Orchestration / evaluation options
# How many example compare images to save (computed metrics will still be produced for all test samples)
save_examples = True
save_examples_count = 10  # total number of example images to save

# Which models to train / evaluate
train_tiny = True
train_vae = True
evaluate_tiny = True
evaluate_vae = True

# Optional pretrained weights (set path to load and optionally skip training)
load_tiny_path = None  # e.g. Path('/path/to/TinyAE.pth')
load_vae_path = None   # e.g. Path('/path/to/VariationalAE.pth')
train_from_scratch = True  # if False and load_* provided, training will be skipped

# Classification merge options (optional)
# predictions_root = Path(r'/path/to/classifier/results_final')
# merged_out_dir = results_dir / 'merged_predictions'

# Ensure directories exist
output_root.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)
examples_dir.mkdir(parents=True, exist_ok=True)
if use_disk_cache:
    disk_cache_dir.mkdir(parents=True, exist_ok=True)

print('Configuration loaded')
print('Results will be saved to:', results_dir)
print('Device:', device)


In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
# Imports
import time, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from skimage.metrics import structural_similarity as ssim
import scipy.ndimage as ndi
from tqdm import tqdm
print('Imports OK')

In [ ]:
# Dataset and loader helper
class CXRBinaryDataset(Dataset):
    """Dataset for CXR images with optional caching. Expects dataframe with columns: filename, subject_id, study_id"""
    def __init__(self, df, base_dir, image_size=224, transform=None, use_mem_cache=True, disk_cache_dir=None):
        self.df = df.reset_index(drop=True)
        self.base_dir = base_dir
        self.image_size = image_size
        self.use_mem_cache = use_mem_cache
        self.disk_cache_dir = str(disk_cache_dir) if disk_cache_dir is not None else None
        self._mem_cache = {} if use_mem_cache else None
        self.transform = transform or transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])
    def __len__(self):
        return len(self.df)
    def _disk_cache_path(self, subject_id, study_id, filename):
        safe = filename.replace('/', '_').replace('//', '_')
        sub = os.path.join(self.disk_cache_dir, f'p{subject_id[:2]}', f'p{subject_id}')
        os.makedirs(sub, exist_ok=True)
        return os.path.join(sub, f's{study_id}_{safe}.pt')
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = str(row['filename'])
        subject_id = str(row.get('subject_id', ''))
        study_id = str(row.get('study_id', ''))
        patient_prefix = subject_id[:2] if subject_id else ''
        img_path = os.path.join(self.base_dir, f'p{patient_prefix}', f'p{subject_id}', f's{study_id}', filename)
        cache_key = f'{subject_id}/{study_id}/{filename}'
        if self._mem_cache is not None and cache_key in self._mem_cache:
            img_t = self._mem_cache[cache_key]
        else:
            img_t = None
            if self.disk_cache_dir:
                try:
                    cache_file = self._disk_cache_path(subject_id, study_id, filename)
                    if os.path.exists(cache_file):
                        img_t = torch.load(cache_file)
                except Exception:
                    img_t = None
            if img_t is None:
                if os.path.exists(img_path):
                    try:
                        img = Image.open(img_path).convert('L')
                    except Exception:
                        img = Image.new('L', (self.image_size, self.image_size), color=0)
                else:
                    img = Image.new('L', (self.image_size, self.image_size), color=0)
                img_t = self.transform(img)
                if self.disk_cache_dir:
                    try:
                        cache_file = self._disk_cache_path(subject_id, study_id, filename)
                        torch.save(img_t.cpu(), cache_file)
                    except Exception:
                        pass
            if self._mem_cache is not None:
                self._mem_cache[cache_key] = img_t
        meta = {'filename': filename, 'subject_id': subject_id, 'study_id': study_id}
        return img_t, torch.tensor(0.0), meta
    
    

def collate_keep_meta(batch):
    # keep meta as a list of per-sample dicts instead of collating into dict-of-lists
    imgs, targets, metas = zip(*batch)
    imgs = torch.stack(imgs, dim=0)
    targets = torch.stack(targets, dim=0)
    return imgs, targets, list(metas)

def make_loaders(train_csv, val_csv, test_csv, base_dir, bs=32, img_size=224, num_workers=0, use_mem_cache=True, disk_cache_dir=None):
    df_train = pd.read_csv(train_csv)
    df_val = pd.read_csv(val_csv)
    df_test = pd.read_csv(test_csv)
    train_ds = CXRBinaryDataset(df_train, base_dir, image_size=img_size, use_mem_cache=use_mem_cache, disk_cache_dir=disk_cache_dir)
    val_ds   = CXRBinaryDataset(df_val,   base_dir, image_size=img_size, use_mem_cache=use_mem_cache, disk_cache_dir=disk_cache_dir)
    test_ds  = CXRBinaryDataset(df_test,  base_dir, image_size=img_size, use_mem_cache=use_mem_cache, disk_cache_dir=disk_cache_dir)
    pin = torch.cuda.is_available()
    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=num_workers, pin_memory=pin, collate_fn=collate_keep_meta)
    val_loader   = DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=num_workers, pin_memory=pin, collate_fn=collate_keep_meta)
    test_loader  = DataLoader(test_ds,  batch_size=bs, shuffle=False, num_workers=num_workers, pin_memory=pin, collate_fn=collate_keep_meta)
    return train_loader, val_loader, test_loader, train_ds, val_ds, test_ds


In [ ]:
# Models
class TinyAE(nn.Module):
    def __init__(self, latent=128):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(1, 32, 4, 2, 1), nn.ReLU(inplace=True),
            nn.Conv2d(32,64, 4, 2, 1), nn.ReLU(inplace=True),
            nn.Conv2d(64,128,4, 2, 1), nn.ReLU(inplace=True),
            nn.Conv2d(128,256,4,2,1),  nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(256*14*14, latent)
        )
        self.dec = nn.Sequential(
            nn.Linear(latent, 256*14*14),
            nn.ReLU(inplace=True),
            nn.Unflatten(1, (256,14,14)),
            nn.ConvTranspose2d(256,128,4,2,1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128,64, 4,2,1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 4,2,1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 1,  4,2,1), nn.Tanh()
        )
    def forward(self, x):
        z = self.enc(x)
        out = self.dec(z)
        return out, z

class VariationalAE(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(1, 32, 4, 2, 1), nn.ReLU(True),
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(True),
            nn.Conv2d(64, 128, 4, 2, 1), nn.ReLU(True),
            nn.Flatten(),
        )
        self.fc_mu = nn.Linear(128 * 28 * 28, latent_dim)
        self.fc_logvar = nn.Linear(128 * 28 * 28, latent_dim)
        self.dec_fc = nn.Linear(latent_dim, 128 * 28 * 28)
        self.dec = nn.Sequential(
            nn.Unflatten(1, (128, 28, 28)),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(True),
            nn.ConvTranspose2d(32, 1, 4, 2, 1), nn.Tanh(),
        )
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    def forward(self, x):
        h = self.enc(x)
        mu, logvar = self.fc_mu(h), self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)
        x_hat = self.dec(self.dec_fc(z))
        return x_hat, (mu, logvar, z)

In [ ]:
def train_autoencoder(model, train_loader, val_loader, epochs=10, lr=1e-3, device=None, model_name=None,
                      recon_tol=0.1, save_dir=None,
                      early_stopping=False, es_patience=5, es_min_delta=0.0, es_restore_best=True):
    """
    Added early_stopping (default False).
    - es_patience: epochs with no improvement before stopping
    - es_min_delta: minimum improvement to reset patience (absolute)
    - es_restore_best: if True, reload best weights before returning
    """
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best_val = float('inf')
    best_saved_path = None
    best_state = None
    no_improve = 0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    name = model_name or model.__class__.__name__
    for ep in range(1, epochs+1):
        model.train()
        tr_loss_sum = 0.0
        tr_correct = 0
        tr_pixels = 0
        for x, _, _ in tqdm(train_loader, desc=f'Train {name} {ep}'):
            x = x.to(device)
            opt.zero_grad()
            out, _ = model(x)
            loss = F.mse_loss(out, x)
            loss.backward()
            opt.step()
            tr_loss_sum += loss.item() * x.size(0)
            with torch.no_grad():
                tr_correct += (torch.abs(out - x) < recon_tol).sum().item()
                tr_pixels += x.numel()
        tr_loss = tr_loss_sum / max(1, len(train_loader.dataset))
        tr_acc = tr_correct / max(1, tr_pixels) if tr_pixels>0 else float('nan')
        # validation
        model.eval()
        val_loss_sum = 0.0
        val_correct = 0
        val_pixels = 0
        with torch.no_grad():
            for x, _, _ in val_loader:
                x = x.to(device)
                out, _ = model(x)
                l = F.mse_loss(out, x)
                val_loss_sum += l.item() * x.size(0)
                val_correct += (torch.abs(out - x) < recon_tol).sum().item()
                val_pixels += x.numel()
        val_loss = val_loss_sum / max(1, len(val_loader.dataset))
        val_acc = val_correct / max(1, val_pixels) if val_pixels>0 else float('nan')
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(val_acc)
        print(f'Epoch {ep:02d} | train MSE: {tr_loss:.5f} | val MSE: {val_loss:.5f} | train acc: {tr_acc:.4f} | val acc: {val_acc:.4f}')

        # check improvement
        improved = (val_loss < best_val - es_min_delta)
        if improved:
            best_val = val_loss
            no_improve = 0
            # save best
            if save_dir is not None:
                try:
                    best_saved_path = os.path.join(save_dir, f'{name}_best.pth')
                    torch.save(model.state_dict(), best_saved_path)
                except Exception as e:
                    print('Could not save best model:', e)
            else:
                # keep in-memory copy
                try:
                    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                except Exception:
                    best_state = None
        else:
            no_improve += 1

        # legacy saving of any improvement (keeps compatibility)
        if val_loss < best_val and save_dir is not None:
            try:
                torch.save(model.state_dict(), os.path.join(save_dir, f'{name}.pth'))
            except Exception as e:
                print('Could not save model:', e)

        # early stopping check
        if early_stopping and (no_improve >= es_patience):
            print(f'Early stopping at epoch {ep} (no improvement for {no_improve} epochs).')
            break

    # save plots
    if save_dir is not None:
        try:
            fig, ax = plt.subplots(1,2, figsize=(12,4))
            ax[0].plot(history['train_loss'], label='train_loss')
            ax[0].plot(history['val_loss'], label='val_loss')
            ax[0].legend(); ax[0].set_title(f'{name} Loss')
            ax[1].plot(history['train_acc'], label='train_acc')
            ax[1].plot(history['val_acc'], label='val_acc')
            ax[1].legend(); ax[1].set_title(f'{name} Recon Acc')
            fig.tight_layout()
            fig_path = os.path.join(save_dir, f'{name}_loss_acc.png')
            fig.savefig(fig_path)
            plt.close(fig)
            print('Saved training curves ->', fig_path)
        except Exception as e:
            print('Could not save training plots:', e)

    # restore best if requested
    if es_restore_best:
        if save_dir is not None and best_saved_path is not None and os.path.exists(best_saved_path):
            try:
                model.load_state_dict(torch.load(best_saved_path, map_location=device))
            except Exception:
                pass
        elif best_state is not None:
            try:
                model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
            except Exception:
                pass

    return model, history

def train_variational_autoencoder(model, train_loader, val_loader, epochs=10, lr=1e-3, device=None,
                                  model_name=None, save_dir=None,
                                  early_stopping=False, es_patience=5, es_min_delta=0.0, es_restore_best=True):
    """
    Added early_stopping (default False) for VAE training as well.
    """
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'val_loss': []}
    name = model_name or model.__class__.__name__
    best_val = float('inf')
    best_saved_path = None
    best_state = None
    no_improve = 0
    for ep in range(1, epochs+1):
        model.train()
        total = 0.0
        for x, _, _ in tqdm(train_loader, desc=f'Train {name} {ep}'):
            x = x.to(device)
            x_hat, (mu, logvar, _) = model(x)
            recon_loss = F.mse_loss(x_hat, x)
            kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
            loss = recon_loss + 0.001 * kl
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item()*x.size(0)
        train_loss = total / max(1, len(train_loader.dataset))
        # val
        model.eval()
        val_total = 0.0
        with torch.no_grad():
            for x, _, _ in val_loader:
                x = x.to(device)
                x_hat, (mu, logvar, _) = model(x)
                recon_loss = F.mse_loss(x_hat, x)
                kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
                val_total += (recon_loss + 0.001*kl).item()*x.size(0)
        val_loss = val_total / max(1, len(val_loader.dataset))
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        print(f'Epoch {ep:02d} | train loss: {train_loss:.5f} | val loss: {val_loss:.5f}')

        # check improvement
        improved = (val_loss < best_val - es_min_delta)
        if improved:
            best_val = val_loss
            no_improve = 0
            if save_dir is not None:
                try:
                    best_saved_path = os.path.join(save_dir, f'{name}_best.pth')
                    torch.save(model.state_dict(), best_saved_path)
                except Exception as e:
                    print('Could not save best VAE model:', e)
            else:
                try:
                    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                except Exception:
                    best_state = None
        else:
            no_improve += 1

        if val_loss < best_val and save_dir is not None:
            try:
                torch.save(model.state_dict(), os.path.join(save_dir, f'{name}.pth'))
            except Exception as e:
                print('Could not save VAE model:', e)

        if early_stopping and (no_improve >= es_patience):
            print(f'Early stopping VAE at epoch {ep} (no improvement for {no_improve} epochs).')
            break

    # save plots
    if save_dir is not None:
        try:
            plt.figure(figsize=(6,4))
            plt.plot(history['train_loss'], label='train_loss')
            plt.plot(history['val_loss'], label='val_loss')
            plt.legend(); plt.title(f'{name} Loss')
            p = os.path.join(save_dir, f'{name}_loss.png')
            plt.savefig(p); plt.close(); print('Saved VAE loss ->', p)
        except Exception as e:
            print('Could not save VAE plot:', e)

    # restore best if requested
    if es_restore_best:
        if save_dir is not None and best_saved_path is not None and os.path.exists(best_saved_path):
            try:
                model.load_state_dict(torch.load(best_saved_path, map_location=device))
            except Exception:
                pass
        elif best_state is not None:
            try:
                model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
            except Exception:
                pass

    return model, history

In [ ]:
# Evaluation utilities and helpers
def mse_np(x,y): return float(((x-y)**2).mean())
def mae_np(x,y): return float(np.abs(x-y).mean())
def psnr_np(x,y, data_range=2.0):
    mse = ((x-y)**2).mean()
    if mse == 0: return float('inf')
    return 20 * math.log10(data_range) - 10 * math.log10(float(mse))
def ssim_np(x,y):
    try: return float(ssim(x,y, data_range=2.0))
    except Exception: return float('nan')
def edge_diff_np(x,y):
    sx = np.hypot(ndi.sobel(x, 0), ndi.sobel(x, 1))
    sy = np.hypot(ndi.sobel(y, 0), ndi.sobel(y, 1))
    return float(np.abs(sx - sy).mean())

def tensor_to_numpy_img(tensor):
    arr = tensor.detach().cpu().numpy()
    if arr.ndim == 3:
        arr = arr[0]
    arr = (arr + 1.0) / 2.0
    arr = np.clip(arr, 0.0, 1.0)
    return (arr * 255).astype(np.uint8)

def save_compare_image(orig_np, recon_np, out_path):
    im1 = Image.fromarray(orig_np).convert('L')
    im2 = Image.fromarray(recon_np).convert('L')
    new = Image.new('L', (im1.width + im2.width, im1.height))
    new.paste(im1, (0,0))
    new.paste(im2, (im1.width,0))
    new.save(out_path)

In [ ]:
def evaluate_and_save(models_dict, loader, device, out_dir, max_samples=None, save_examples_limit=None):
    import os, re, torch, numpy as np
    from tqdm import tqdm
    os.makedirs(out_dir, exist_ok=True)

    rows = []
    processed = 0
    saved_examples = 0
    save_failures = []

    for batch in tqdm(loader, desc='Evaluate'):
        x, _, meta = batch

        # reconstruct per-sample meta
        meta_list = []
        if isinstance(meta, dict):
            vals = list(meta.values())
            if vals and isinstance(vals[0], (list, tuple, np.ndarray, torch.Tensor)):
                L = len(vals[0])
                for i in range(L):
                    item = {}
                    for k, v in meta.items():
                        if isinstance(v, (list, tuple, np.ndarray, torch.Tensor)):
                            try:
                                item[k] = v[i]
                            except Exception:
                                item[k] = v
                        else:
                            item[k] = v
                    meta_list.append(item)
            else:
                meta_list = [meta]
        else:
            meta_list = list(meta)

        x = x.to(device)
        recons = {}
        for name, m in models_dict.items():
            try:
                m = m.to(device)
                m.eval()
                out = m(x)
                xhat = out[0] if isinstance(out, tuple) else out
                recons[name] = xhat.detach().cpu().numpy()
            except Exception:
                # fallback zeros with same shape
                recons[name] = np.zeros_like(x.detach().cpu().numpy())

        x_np = x.detach().cpu().numpy()
        B = x_np.shape[0]

        for i in range(B):
            meta_i = meta_list[i] if i < len(meta_list) else {}
            # filename extraction + sanitize
            fname_raw = None
            if isinstance(meta_i, dict):
                fname_raw = meta_i.get('filename', None)
            else:
                fname_raw = meta_i
            if isinstance(fname_raw, (list, tuple, np.ndarray, torch.Tensor)):
                try:
                    fname = str(fname_raw[0])
                except Exception:
                    fname = str(fname_raw)
            else:
                fname = str(fname_raw) if fname_raw is not None else ''
            if not fname or fname.lower() in ('none', 'nan'):
                fname = f"ex_{processed:06d}"
            fname = re.sub(r'[<>:\"/\\|?*\s]+', '_', fname)

            subj = meta_i.get('subject_id', '') if isinstance(meta_i, dict) else ''
            study = meta_i.get('study_id', '') if isinstance(meta_i, dict) else ''
            row = {'filename': fname, 'subject_id': subj, 'study_id': study}

            orig = tensor_to_numpy_img(torch.from_numpy(x_np[i]))

            for mname, rarr in recons.items():
                recon_img = tensor_to_numpy_img(torch.from_numpy(rarr[i]))
                orig_f = (orig.astype(np.float32) / 127.5) - 1.0
                recon_f = (recon_img.astype(np.float32) / 127.5) - 1.0

                row[f'{mname}_mse'] = mse_np(orig_f, recon_f)
                row[f'{mname}_mae'] = mae_np(orig_f, recon_f)
                row[f'{mname}_psnr'] = psnr_np(orig_f, recon_f)
                row[f'{mname}_ssim'] = ssim_np(orig_f, recon_f)
                row[f'{mname}_edge_diff'] = edge_diff_np(orig_f, recon_f)

                out_name = f"{fname}_{mname}_compare.jpg"
                out_path = os.path.join(out_dir, out_name)

                # only write example images up to save_examples_limit (None -> unlimited)
                if (save_examples_limit is None) or (saved_examples < save_examples_limit):
                    try:
                        save_compare_image(orig, recon_img, out_path)
                        row[f'{mname}_example'] = out_name
                        saved_examples += 1
                    except Exception as e:
                        save_failures.append((out_path, str(e)))
                        row[f'{mname}_example'] = ''
                else:
                    row[f'{mname}_example'] = ''

            rows.append(row)
            processed += 1

            # legacy processing limit
            if (max_samples is not None) and (processed >= max_samples):
                break

        if (max_samples is not None) and (processed >= max_samples):
            break

    if save_failures:
        print(f"Warning: {len(save_failures)} example images failed to save (first: {save_failures[0][0]})")
    return rows


In [ ]:
# Orchestration: run training and evaluation
# Run this cell after all previous cells have been executed

def run_all(save_examples=True,
            save_examples_count=10,
            train_tiny=True,
            train_vae=True,
            load_tiny_path=None,
            load_vae_path=None,
            train_from_scratch=True,
            evaluate_tiny=True,
            evaluate_vae=True,
            # early stopping override (defaults to config above)
            early_stopping_cfg=None,
            es_patience_cfg=None,
            es_min_delta_cfg=None,
            es_restore_best_cfg=None):
    # resolve early-stopping params (use explicit args if provided, else global config)
    early_stopping_use = early_stopping_cfg if (early_stopping_cfg is not None) else globals().get('early_stopping', False)
    es_patience_use   = es_patience_cfg   if (es_patience_cfg   is not None) else globals().get('es_patience', 5)
    es_min_delta_use  = es_min_delta_cfg  if (es_min_delta_cfg  is not None) else globals().get('es_min_delta', 1e-4)
    es_restore_use    = es_restore_best_cfg if (es_restore_best_cfg is not None) else globals().get('es_restore_best', True)

    # optional sampling to quickly test pipeline
    if 0.0 < sample_fraction < 1.0:
        print(f'Sampling {sample_fraction*100:.2f}% of each split for quick run')
        for csv_path, outname in [(train_csv, 'train_subset.csv'), (val_csv, 'val_subset.csv'), (test_csv, 'test_subset.csv')]:
            df = pd.read_csv(csv_path)
            df_s = df.sample(frac=sample_fraction, random_state=subset_random_seed).reset_index(drop=True)
            outp = results_dir / outname
            df_s.to_csv(outp, index=False)
        t_csv = str(results_dir / 'train_subset.csv')
        v_csv = str(results_dir / 'val_subset.csv')
        te_csv = str(results_dir / 'test_subset.csv')
    else:
        t_csv, v_csv, te_csv = train_csv, val_csv, test_csv

    train_loader, val_loader, test_loader, tr_ds, v_ds, te_ds = make_loaders(t_csv, v_csv, te_csv, base_dir, bs=batch_size, img_size=img_size, num_workers=num_workers, use_mem_cache=use_mem_cache, disk_cache_dir=(str(disk_cache_dir) if use_disk_cache else None))

    # instantiate models
    tiny = TinyAE(latent=128)
    vae = VariationalAE(latent_dim=128)

    # optionally load pretrained weights
    if load_tiny_path:
        try:
            sd = torch.load(load_tiny_path, map_location=device)
            tiny.load_state_dict(sd)
            print('Loaded TinyAE weights from', load_tiny_path)
        except Exception as e:
            print('Could not load TinyAE weights:', e)
    if load_vae_path:
        try:
            sd = torch.load(load_vae_path, map_location=device)
            vae.load_state_dict(sd)
            print('Loaded VariationalAE weights from', load_vae_path)
        except Exception as e:
            print('Could not load VAE weights:', e)

    # train tiny
    if train_tiny:
        print('Training TinyAE')
        tiny, h_tiny = train_autoencoder(
            tiny, train_loader, val_loader,
            epochs=tiny_epochs, lr=lr_tiny,
            device=device, model_name='TinyAE',
            recon_tol=recon_tol, save_dir=str(results_dir),
            early_stopping=early_stopping_use,
            es_patience=es_patience_use,
            es_min_delta=es_min_delta_use,
            es_restore_best=es_restore_use
        )
    else:
        print('Skipping TinyAE training')

    # train vae
    if train_vae:
        print('Training VAE')
        vae, h_vae = train_variational_autoencoder(
            vae, train_loader, val_loader,
            epochs=vae_epochs, lr=lr_vae,
            device=device, model_name='VariationalAE', save_dir=str(results_dir),
            early_stopping=early_stopping_use,
            es_patience=es_patience_use,
            es_min_delta=es_min_delta_use,
            es_restore_best=es_restore_use
        )
    else:
        print('Skipping VAE training')
    # build models dict for evaluation based on flags
    models = {}
    if evaluate_tiny:
        models['TinyAE'] = tiny
    if evaluate_vae:
        models['VariationalAE'] = vae

    if not models:
        print('No models selected for evaluation; exiting')
        return

    print('Evaluating on test set and saving example images...')
    save_limit = (save_examples_count if save_examples else 0)
    rows = evaluate_and_save(models, test_loader, device, out_dir=str(examples_dir), save_examples_limit=save_limit)
    if rows:
        df = pd.DataFrame(rows)
        csv_out = results_dir / 'results_metrics.csv'
        df.to_csv(csv_out, index=False)
        print('Saved per-image metrics ->', csv_out)
    else:
        print('No rows returned from evaluation')


# Example: run training+evaluation but save only 10 example images while computing metrics for all test samples
run_all(save_examples=True, save_examples_count=10, train_tiny=True, train_vae=True)
